# Initialization

In [1]:
# read the data
import matplotlib.pyplot as plt
import numpy as np
#import pandas as pd
import matplotlib

import mne     
import os
from mne.preprocessing import ICA
%matplotlib qt

In [2]:
# understand the 
import sys
sys.path.append(os.path.abspath('..'))
from utils import identify_bad_channels,remove_epochs_with_bad_mmn_channels, Average_in_trials_in_time_window, remove_epochs_with_n_bad_channels ,remove_or_interpolate_epochs_with_n_bad_chn
import importlib
import utils 
importlib.reload(utils)

#from utils import remove_or_interpolate_epochs


<module 'utils' from 'e:\\ecf-exp2-notif-mmn\\analysis\\utils.py'>

In [3]:
from mne.utils import set_config
from mne.utils import get_config
set_config('MNE_USE_CUDA', 'true')
mne.set_log_level('WARNING')  # Options: 'DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL'
# Check that the config is set
print(get_config('MNE_USE_CUDA')) 
#mne.set_memmap_min_size('100M')

true


# Data Reading 

In [4]:
data_folder = r"D:\Work_data\Notification_evoked_response\Data_at_IITD/bids_data/"
subjects = [sub for sub in os.listdir(data_folder) if (("sub-" in sub) & (os.path.isdir(os.path.join(data_folder,sub))))]
print(subjects)
data_type = "eeg"
 # it is useful to scroll eeg data 
B1_MMN_erp_evoked_dict  = {}
B1_devi_eps_evoked_dict  = {} 
B1_sta_eps_evoked_dict  = {}                            
B2_devi_eps_evoked_dict = {}
B2_sta_eps_evoked_dict = {}
B2_MMN_erp_evoked_dict = {}
epoch_per_num = {}
epoch_per_num["subject Dropped"] = []


['sub-sd011', 'sub-sd012', 'sub-sd013', 'sub-sd014', 'sub-sd015', 'sub-sd016', 'sub-sd017', 'sub-sd018', 'sub-sd019', 'sub-sd020', 'sub-sd021', 'sub-sd022', 'sub-sd023', 'sub-sd024', 'sub-sd025', 'sub-sd026', 'sub-sd027', 'sub-sd028', 'sub-sd029', 'sub-sd030', 'sub-sd031', 'sub-sd032', 'sub-sd033', 'sub-sd034', 'sub-sd035', 'sub-sd036', 'sub-sd037', 'sub-sd038', 'sub-sd039', 'sub-sd040', 'sub-sd041', 'sub-sd042', 'sub-sd043', 'sub-sd044', 'sub-sd045', 'sub-sd046', 'sub-sd047', 'sub-sd048', 'sub-sd049', 'sub-sd050', 'sub-sd051', 'sub-sd052', 'sub-sd053', 'sub-sd054', 'sub-sd055', 'sub-sd056', 'sub-sd057', 'sub-sd058', 'sub-sd059', 'sub-sd060', 'sub-sd061', 'sub-sd062', 'sub-sd063', 'sub-sd064', 'sub-sd065', 'sub-sd066', 'sub-sd067']


# Visual inspections and marking bad channels 

In [5]:
sub = 'sub-sd034'
path = os.path.join(data_folder,sub,data_type)
    
for eeg in os.listdir(path):
        if eeg[-5:] == ".vhdr":
           
            raw = mne.io.read_raw_brainvision(os.path.join(path,eeg), preload=True)
            #raw = mne.add_reference_channels(raw, ref_channels=["Cz"])
            raw.set_montage("easycap-M1")
            raw.plot(scalings=dict(eeg=40e-6) )

In [6]:
raw.info

Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,67 points
Good channels,64 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,1000.00 Hz
Highpass,0.00 Hz
Lowpass,500.00 Hz


# Pre processing

In [7]:
raw.set_montage("easycap-M1")
#  raw.interpolate_bads(reset_bads=True)
raw.set_eeg_reference(ref_channels=['TP9','TP10'])
raw.resample(512)
            # filter 
raw.filter(l_freq=0.1,h_freq=30 )
raw.plot(scalings=dict(eeg=40e-6))

# ICA 

In [8]:
#ICA  
ica = ICA(n_components=None, random_state=97)
ica.fit(raw)
print("ICA Completed ...")
ica.plot_components() 

ICA Completed ...


[<MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 585x260 with 3 Axes>]

In [9]:
eog_channels = ['Fp1', 'Fp2',"AF7","AF8",'F7', 'F8']  # Adjust these to match your channel names
eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name=eog_channels)
print(eog_indices, eog_scores)

[1, 0, 12, 8] [array([ 0.14106705,  0.96923191,  0.00254603,  0.00369862, -0.04529094,
        0.05463809, -0.09179743,  0.10472624, -0.11461335,  0.12402663,
        0.07920835, -0.07885374, -0.13471407, -0.00568499,  0.0444277 ,
       -0.03139111,  0.03563874, -0.11453329,  0.0679754 , -0.04076459,
       -0.0407997 ,  0.137228  , -0.00611071,  0.01231269,  0.01377888,
       -0.02451358, -0.04611145,  0.00552798, -0.08541941,  0.04309479,
       -0.0039216 , -0.12043382,  0.0089443 , -0.05084745, -0.00417475,
        0.065748  ,  0.00352894,  0.01371431,  0.07566428, -0.02135189,
        0.01769617, -0.01725854, -0.02534911, -0.02287705,  0.01555234,
       -0.03472415, -0.0807628 , -0.00928437,  0.01224243, -0.01915902,
       -0.0188679 ,  0.05782765,  0.01132856,  0.02353808,  0.00885957,
        0.05565799,  0.02694424, -0.07761404, -0.01794211,  0.03669427,
       -0.0144403 ,  0.01118928, -0.02837026]), array([ 1.11728524e-01,  9.80330945e-01, -6.66164069e-03,  2.65418907e-03

In [10]:
ica.exclude

[1]

In [11]:
# Update the ICA 
ica.exclude  = [1]
ica.apply(raw) 
raw.plot(scalings=dict(eeg=50e-6))

# Interpolate bad

In [12]:
raw.interpolate_bads(reset_bads=True)

C:\Users\PRAKASH\AppData\Local\Temp\ipykernel_22112\2287972999.py:1: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw.interpolate_bads(reset_bads=True)


Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,67 points
Good channels,64 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,512.00 Hz
Highpass,0.10 Hz
Lowpass,30.00 Hz


# Epoching

In [13]:
events,event_dict = mne.events_from_annotations(raw)
print(events,event_dict)


[[      0       0   10010]
 [      0       0   10009]
 [      0       0   10010]
 ...
 [1210257       0   10007]
 [1211238       0   10006]
 [1211238       0   10006]] {'Stimulus/S   11': 10001, 'Stimulus/S   12': 10002, 'Stimulus/S   13': 10003, 'Stimulus/S   14': 10004, 'Stimulus/S   15': 10005, 'Stimulus/S   21': 10006, 'Stimulus/S   24': 10007, 'Stimulus/S   25': 10008, 'Stimulus/S10001': 10009, 'Stimulus/S99999': 10010}


In [14]:
# Create Epochs without droping the eye blink event
event_id = {
                'S14': event_dict['Stimulus/S   14'],
                'S15': event_dict['Stimulus/S   15'],
                'S24': event_dict['Stimulus/S   24'],
                'S25': event_dict['Stimulus/S   25'],
                # 'S34': event_dict['Stimulus/S   34'],
                # 'S35': event_dict['Stimulus/S   35'],
                }
epochs = mne.Epochs(raw, events, event_id=event_id,baseline=(None, 0) , tmin=-0.300, tmax=0.900, preload=True,event_repeated='merge')
# epochs.drop_channels(['GSR', 'PPG'])

In [15]:
epochs.plot(events=events,
                     scalings=dict(eeg=20e-6),
                     n_epochs=50,)


# Drop or Interpolate Bad epochs  

In [16]:
cleaned_epochs = remove_or_interpolate_epochs_with_n_bad_chn(epochs,  bad_chn_threshold = 6, amplitude_threshold=80e-6)    

e:\ecf-exp2-notif-mmn\analysis\utils.py:188: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs.get_data()


Epoch index 0 ; Interpolating channels ['F7']
Epoch index 1 ; Interpolating channels ['Pz', 'AF7', 'C5', 'AF8']
Epoch index 3 ; Interpolating channels ['CP2']
Epoch index 4 ; Interpolating channels ['CP2', 'AF7', 'F5', 'C5']
Epoch index 7 ; Interpolating channels ['FT10']
Epoch index 10 ; Interpolating channels ['Cz']
Epoch index 11 ; Interpolating channels ['P5', 'Cz']
Epoch index 12 ; Interpolating channels ['Cz']
Epoch index 14 ; Interpolating channels ['Cz']
Epoch index 17 ; Interpolating channels ['T8', 'Cz']
Epoch index 18 ; Interpolating channels ['Cz']
Epoch index 21 ; Interpolating channels ['AF8']
Epoch index 24 ; Interpolating channels ['C5']
Epoch index 34 ; Interpolating channels ['AF8']
Epoch index 37 ; Interpolating channels ['AF8']
Epoch index 38 ; Interpolating channels ['Pz', 'AF8']
Epoch index 39 ; Interpolating channels ['AF8']
Epoch index 41 ; Interpolating channels ['Pz']
Epoch index 42 ; Interpolating channels ['Pz']
Epoch index 44 ; Interpolating channels ['AF8'

e:\ecf-exp2-notif-mmn\analysis\utils.py:217: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(good_epochs)


In [18]:
print(f"Original number of epochs: {len(epochs)}")
print(f"Number of epochs after removal: {len(cleaned_epochs)}")
cleaned_epochs.plot()

Original number of epochs: 1800
Number of epochs after removal: 1718


# Drop the mannual marked bad Epochs 

In [22]:
cleaned_epochs.drop_bad()
print(f"Original number of epochs: {len(epochs)}")
print(f"Number of epochs after removal: {len(cleaned_epochs)}")
cleaned_epochs.plot()

Original number of epochs: 1800
Number of epochs after removal: 1584


In [20]:
PN_devi_eps = cleaned_epochs['S15'].average()
BN_sta_eps = cleaned_epochs['S14'].average()

BN_devi_eps = cleaned_epochs['S25'].average() 
PN_sta_eps = cleaned_epochs['S24'].average() 
# CN_devi_eps = cleaned_epochs['S35'].average() 
# CN_sta_eps = cleaned_epochs['S34'].average() 


In [25]:
evokeds = { 
    "SN MMN" : mne.combine_evoked([ PN_devi_eps,PN_sta_eps], weights=[1,-1]),
    'BN MMN':mne.combine_evoked([ BN_devi_eps,BN_sta_eps], weights=[1,-1]),
    # "CN MMN" : mne.combine_evoked([ CN_devi_eps,CN_sta_eps], weights=[1,-1]),
           }

stylesdict ={"SN MMN": {"linewidth": 3,"linestyle":'solid'},
             "BN MMN" : {"linewidth": 3,"linestyle":'solid'},
            #  "CN MMN" : {"linewidth": 3,"linestyle":'solid'},
             }
mne.viz.plot_compare_evokeds(evokeds,styles = stylesdict,
                             truncate_xaxis=False,truncate_yaxis=False,picks=['Cz'],
                             ylim = dict(eeg=[-6.2,6.2]))

[<Figure size 800x600 with 2 Axes>]

# Save the Epochs data 

In [23]:
folder = r"E:\ecf-exp2-notif-mmn\data\Processed"
cleaned_epochs.reset_drop_log_selection()
cleaned_epochs.save(f'{os.path.join(folder,sub)}-epo.fif', overwrite=True)

# Read data and verify the results 

In [ ]:
re_epochs = mne.read_epochs(f'{os.path.join(folder,sub)}-epo.fif', preload=True)

print(f"Number of epochs after re_epochs read: {len(re_epochs)}")

Number of epochs after re_epochs read: 1639


In [27]:
re_PN_devi_eps = re_epochs['S15'].average()
re_BN_sta_eps = re_epochs['S14'].average()

re_BN_devi_eps = re_epochs['S25'].average() 
re_PN_sta_eps = re_epochs['S24'].average() 
# re_CN_devi_eps = re_epochs['S35'].average() 
# re_CN_sta_eps = re_epochs['S34'].average() 

In [28]:
re_evokeds = { 
    "SN MMN" : mne.combine_evoked([ re_PN_devi_eps,re_PN_sta_eps], weights=[1,-1]),
    'BN MMN':mne.combine_evoked([ re_BN_devi_eps,re_BN_sta_eps], weights=[1,-1]),
    # "CN MMN" : mne.combine_evoked([ re_CN_devi_eps,re_CN_sta_eps], weights=[1,-1]),
           }

stylesdict ={"SN MMN": {"linewidth": 3,"linestyle":'solid'},
             "BN MMN" : {"linewidth": 3,"linestyle":'solid'},
            #  "CN MMN" : {"linewidth": 3,"linestyle":'solid'},
             }
mne.viz.plot_compare_evokeds(re_evokeds,styles = stylesdict,
                             truncate_xaxis=False,truncate_yaxis=False,picks=['Cz'],
                             ylim = dict(eeg=[-6.2,6.2]))

[<Figure size 800x600 with 2 Axes>]